# Session 11 · Homework Solutions (Teacher Copy) — Thresholds & Multi-Class

**Machine Learning Foundations · Sanketana School of Code**

Worked solution with commentary. Part A rehearses the S12–13 reasoning: a strict rule pushes the threshold **up**, trading false passes for missed passers. Part B gives a ~0.89-accuracy three-class model.

**Acceptable variation:** any threshold defended by the correct *named mistake* passes; band cutoffs may shift a little if the student documents them.

## Part A · Choose a threshold with a reason

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

students = pd.read_csv("../../../datasets/anchor/student_habits.csv")
habits = ["study_hours_per_week", "attendance_pct", "sleep_hours_per_night",
          "screen_time_hours_per_day", "practice_sessions_per_week"]
X = students[habits].values
y = students["passed"].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
model = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), y_train)
p_pass = model.predict_proba(scaler.transform(X_test))[:, 1]

print(f"{'cut':>4} {'missed-passers':>15} {'false-passes':>14}")
for t in [0.3, 0.5, 0.7]:
    pred = (p_pass >= t).astype(int)
    missed = int(np.sum((y_test == 1) & (pred == 0)))
    falsep = int(np.sum((y_test == 0) & (pred == 1)))
    print(f"{t:>4} {missed:>15} {falsep:>14}")
# Typical: cut 0.3 -> missed ~1,  false ~10
#          cut 0.5 -> missed ~6,  false ~2
#          cut 0.7 -> missed ~14, false ~1

### ✅ Part A answer

*"'Never falsely tell a student they'll pass' means minimising **false passes**, so I'd raise the threshold to **0.7**, where false passes drop to about 1. The cost is **missed passers**: about 14 real passers are now predicted to fail. I'm choosing to make the 'missed passer' mistake more often in order to almost never make the 'false pass' mistake."*

A choice with **no named mistake and no numbers** does not pass. The direction (strict rule → higher threshold) is the key idea.

## Part B · Three grade bands (multi-class)

In [ ]:
bands = pd.cut(students["test_score"], bins=[0, 50, 75, 100],
               labels=["fail", "pass", "distinction"])
yb = bands.values
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
    X, yb, test_size=0.25, random_state=42, stratify=yb)
scb = StandardScaler().fit(Xb_tr)
multi = LogisticRegression(max_iter=1000).fit(scb.transform(Xb_tr), yb_tr)
print("multi-class test accuracy:", round(multi.score(scb.transform(Xb_te), yb_te), 3))  # ~0.89
row = multi.predict_proba(scb.transform(Xb_te))[0]
print("classes:", list(multi.classes_), " probabilities:", np.round(row, 2))

### ✅ Part B answer

*"The model predicts the class with the highest probability. For this student the probabilities are roughly `distinction 0.04, fail 0.00, pass 0.96`, so it predicts **pass** with about **96%** confidence."* (Exact numbers depend on which student lands first in the test split — grade the *reading*, not the specific decimals.)

**Review talking point for Session 12:** all session we traded two mistakes without proper names. Next session we name them — **false positive** and **false negative** — put them in a **confusion matrix**, and switch to **fraud** data, where 96%-accurate can mean *catching zero fraud*. That's where 'which mistake is worse' becomes an ethics question about *who pays*.